# Topic Modeling with LDA

An unsupervised NLP project that automatically discovers hidden topics from a collection of text documents using **Latent Dirichlet Allocation (LDA)**.



## Project Overview

| Detail | Info |
|--------|------|
| Type | Unsupervised NLP / Topic Modeling |
| Language | Python |
| Libraries | NLTK, Scikit-learn, Regex, Unicodedata |
| Input | Collection of raw text documents |
| Output | Top keywords per discovered topic |



## Pipeline Flow

```
Raw Text Documents
       ↓
Remove Emojis + Clean Text
       ↓
Tokenize + Remove Stopwords + Lemmatize
       ↓
Convert to Bag of Words (CountVectorizer)
       ↓
Train LDA Model
       ↓
Extract Top Keywords per Topic ✅
```

## Import Libraries

| Library | Purpose |
|---------|---------|
| `re` | Regex for text cleaning |
| `string` | Removing punctuation |
| `unicodedata` | Normalizing unicode characters |
| `nltk` | Tokenization, stopwords, lemmatization |
| `CountVectorizer` | Convert text to word count matrix |
| `LatentDirichletAllocation` | LDA topic modeling algorithm |

In [1]:
import re
import string
import unicodedata
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

## Download NLTK Resources

| Download | Used For |
|----------|---------|
| `punkt` | Tokenization |
| `stopwords` | Removing common words |
| `wordnet` | Lemmatization |

In [2]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [3]:
def remove_emojis(text):
    return text.encode("ascii", "ignore").decode()

In [4]:
def clean_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)   # remove URLs
    text = re.sub(r"<.*?>", " ", text)                    # remove HTML tags
    text = re.sub(r"\S+@\S+", " ", text)                  # remove emails
    text = re.sub(r"@\w+|#\w+", " ", text)                # remove mentions and hashtags
    text = re.sub(r"\d+", " ", text)                      # remove numbers
    text = remove_emojis(text)                            # remove emojis
    text = text.translate(str.maketrans("", "", string.punctuation))  # remove punctuation
    text = re.sub(r"\s+", " ", text).strip()              # normalize spaces
    return text

In [5]:
def tokenize(text):
    return word_tokenize(text)

In [6]:
stop_words = set(stopwords.words("english"))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in stop_words and len(t) > 1]

In [7]:
lemmatizer = WordNetLemmatizer()

def lemmatize(tokens):
    return [lemmatizer.lemmatize(t) for t in tokens]

## Convert Text to Numbers (CountVectorizer)

LDA cannot read text directly — it needs numbers.

**CountVectorizer** converts documents into a **word count matrix:**

```
           machine  learning  football  climate
doc1  →  [   1,       1,         0,       0   ]
doc4  →  [   0,       0,         1,       0   ]
doc8  →  [   0,       0,         0,       1   ]
```

- Each row = one document
- Each column = one word
- Each value = how many times that word appears

> 💡 This is called a **Bag of Words** representation.

In [8]:
def vectorize(clean_docs):
    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(clean_docs)
    words = vectorizer.get_feature_names_out()
    return X, words

## Train LDA Model

**LDA (Latent Dirichlet Allocation)** finds hidden topics in the documents.


| Parameter | Value | Meaning |
|-----------|-------|---------|
| `n_components` | `3` | find 3 topics |
| `random_state` | `42` | same result every run |

In [9]:
def train_lda(X, num_topics):
    lda = LatentDirichletAllocation(n_components=num_topics, random_state=42)
    lda.fit(X)
    return lda

## View Top Keywords per Topic

After training, LDA gives each topic a list of the most important words.

**Expected Output:**
```
Topic 1 → learning  machine  deep  image  recognition   (Technology)
Topic 2 → football  tennis   team  match  tournament     (Sports)
Topic 3 → climate   election policy ocean  government    (Politics/Science)
```

In [10]:
def show_topics(lda, words):
    print("\nTopics Found:")
    print("-" * 40)
    for i, topic in enumerate(lda.components_):
        top_words = [words[j] for j in topic.argsort()[-5:]]
        print(f"Topic {i+1} → {' | '.join(top_words)}")

In [13]:
print("Enter documents (type 'done' to finish)")

docs = []
i = 1

while True:
    doc = input(f"Doc {i}: ")
    if doc.lower() == "done":
        break
    if doc:
        docs.append(doc)
        i += 1

clean_docs = []

for doc in docs:
    text = clean_text(doc)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize(tokens)
    clean_docs.append(" ".join(tokens))


num_topics = int(input("Topics: "))

X, words = vectorize(clean_docs)
lda = train_lda(X, num_topics)
show_topics(lda, words)

Enter documents (type 'done' to finish)
Doc 1: Machine learning improves software
Doc 2: Football team won the match
Doc 3: Elections decide political leadership
Doc 4: done
Topics: 2

Topics Found:
----------------------------------------
Topic 1 → match | election | decide | political | leadership
Topic 2 → football | learning | improves | machine | software
